In [1]:
import os
import pandas as pd
import numpy as np
import torch
from torch import nn
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight

In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))

CUDA available: True
Device: Tesla T4


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# 1. Load the Saved Data
EXPORT_DIR = '/content/drive/MyDrive/Finlens/data/splits/'
train_df = pd.read_csv(os.path.join(EXPORT_DIR, 'train_split.csv'))
val_df = pd.read_csv(os.path.join(EXPORT_DIR, 'val_split.csv'))
test_df = pd.read_csv(os.path.join(EXPORT_DIR, 'test_split.csv'))

# Check if data loaded successfully
print(f"Training set loaded: {len(train_df)} rows")

Training set loaded: 7375 rows


In [5]:
# 2. Map Labels to Integers
label_mapping = {
    'COMPLIANT': 0,
    'UNDISCLOSED_FEES': 1,
    'KDPA_EXCESSIVE_DATA_COLLECTION': 2,
    'AGGRESSIVE_COLLECTION': 3,
    'KDPA_CONTACT_ACCESS': 4,
    'KDPA_INADEQUATE_CONSENT': 5
}

train_df['label'] = train_df['violation_category'].map(label_mapping)
val_df['label'] = val_df['violation_category'].map(label_mapping)
test_df['label'] = test_df['violation_category'].map(label_mapping)

In [6]:
# 3. Calculate Class Weights for the Loss Function
classes = np.unique(train_df['label'])
weights = compute_class_weight(class_weight='balanced', 
                               classes=classes, 
                               y=train_df['label'])
# Move weights to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class_weights_tensor = torch.tensor(weights, dtype=torch.float).to(device)

In [7]:
# 4. Initialize LEGAL-BERT Tokenizer and Model
model_name = "nlpaueb/legal-bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6).to(device)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those 

In [8]:
print(train_df.columns.tolist())

['clause_id', 'document_id', 'provider_name', 'document_type', 'source_type', 'raw_text', 'cleaned_clause_text', 'violation_category', 'risk_level', 'violating_span', 'primary_law_reference', 'annotator_notes', 'label']


In [9]:
# 5. Tokenize the Text (Updated with the correct column name)
def tokenize_data(df):
    return tokenizer(
        df['cleaned_clause_text'].astype(str).tolist(), # Updated column name and added .astype(str) for safety
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors='pt'
    )

train_encodings = tokenize_data(train_df)
val_encodings = tokenize_data(val_df)
test_encodings = tokenize_data(test_df)

In [10]:
# 6. Create the PyTorch Dataset
class FinLensDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx].clone().detach() for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = FinLensDataset(train_encodings, train_df['label'].tolist())
val_dataset = FinLensDataset(val_encodings, val_df['label'].tolist())
test_dataset = FinLensDataset(test_encodings, test_df['label'].tolist())

In [11]:
# 7. Define Evaluation Metrics (Macro average for imbalanced classes)
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    acc = accuracy_score(labels, preds)
    
    return {
        'accuracy': acc,
        'f1_macro': f1,
        'precision_macro': precision,
        'recall_macro': recall
    }

In [12]:
# 8. Subclass Trainer to Inject the Weighted Loss
class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Apply the balanced class weights to penalize missing rare violations
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss

In [13]:
# 9. Configure Training Arguments and Execute (Updated)
training_args = TrainingArguments(
    output_dir='./finlens_legal_bert',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=50, 
    eval_strategy="epoch",  
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro", 
    greater_is_better=True
)
trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# Start training!
print("Starting FinLens training...")
trainer.train()

Starting FinLens training...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision Macro,Recall Macro
1,0.529328,0.895668,0.989140,0.165757,0.164857,0.166667
2,0.564411,0.986955,0.989140,0.165757,0.164857,0.166667
3,0.797593,0.777775,0.989975,0.249160,0.331661,0.222222
4,0.530853,0.641265,0.989975,0.265965,0.236975,0.333052


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1844, training_loss=0.7118976990209485, metrics={'train_runtime': 3127.9488, 'train_samples_per_second': 9.431, 'train_steps_per_second': 0.59, 'total_flos': 7762054892544000.0, 'train_loss': 0.7118976990209485, 'epoch': 4.0})

In [14]:
from sklearn.metrics import classification_report

predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(axis=1)
true_labels = predictions.label_ids

print(classification_report(
    true_labels,
    preds,
    target_names=[
        'COMPLIANT',
        'UNDISCLOSED_FEES',
        'KDPA_EXCESSIVE_DATA_COLLECTION',
        'AGGRESSIVE_COLLECTION',
        'KDPA_CONTACT_ACCESS',
        'KDPA_INADEQUATE_CONSENT'
    ],
    digits=4,
    zero_division=0
))

                                precision    recall  f1-score   support

                     COMPLIANT     0.9916    0.9976    0.9946       832
              UNDISCLOSED_FEES     0.0000    0.0000    0.0000         2
KDPA_EXCESSIVE_DATA_COLLECTION     0.5000    1.0000    0.6667         2
         AGGRESSIVE_COLLECTION     0.0000    0.0000    0.0000         2
           KDPA_CONTACT_ACCESS     0.0000    0.0000    0.0000         2
       KDPA_INADEQUATE_CONSENT     0.0000    0.0000    0.0000         1

                      accuracy                         0.9893       841
                     macro avg     0.2486    0.3329    0.2769       841
                  weighted avg     0.9822    0.9893    0.9855       841



In [15]:
# 10. Evaluate on the Test Set
print("\n--- Final Test Set Evaluation ---")
test_results = trainer.evaluate(test_dataset)
print(test_results)


--- Final Test Set Evaluation ---


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro,Precision Macro,Recall Macro
0.530853,0.898797,4,0.989298,0.276879,0.248606,0.332933


{'eval_loss': 0.8987967371940613, 'eval_accuracy': 0.9892984542211652, 'eval_f1_macro': 0.2768790360162439, 'eval_precision_macro': 0.2486061330147352, 'eval_recall_macro': 0.3329326923076923}
